# Tổng hợp Trực quan hóa & Phân tích Insight — Datathon 2026 Gridbreaker
**The GridBreakers · VinTelligence · VinUni DS&AI Club**

---

## Mục lục
| Phần | Nội dung |
|---|---|
| **Phần 1** | Data Profiling (6 biểu đồ) — `00_data_profiling.ipynb` |
| **Phần 2** | EDA Khám phá (4 biểu đồ) — `01_eda_exploratory.ipynb` |
| **Phần 3** | Dashboard Streamlit (9 trang) — `src/app/` |
| **Phần 4** | Dự báo & SHAP (chuỗi A1–G) — `part3_pipeline.ipynb` |
| **Phần 5** | Bảng tham chiếu chéo: Chart nào trả lời câu hỏi kinh doanh nào |
| **Phần 6** | Data Storytelling — Từ dữ liệu đến câu chuyện kinh doanh |
| **Phần 7** | Nguyên tắc thiết kế & Khả năng tái lập |
| **Phần 8** | Hình ảnh bổ sung khác |

> **Mỗi biểu đồ có 3 phần:**
> 1. **Trực quan hóa cái gì** — mục đích, dữ liệu đầu vào
> 2. **Plan** — cách tạo biểu đồ, xử lý dữ liệu, chọn loại biểu đồ
> 3. **Insight** — cách đọc biểu đồ, ý nghĩa và kết luận kinh doanh


---
## Phần 1: Data Profiling — Trực quan hóa Hồ sơ Dữ liệu
**Nguồn:** `notebooks/02_exploration/00_data_profiling.ipynb`

**Mục tiêu:** Hiểu rõ shape, dtype, phân phối, và các điểm bất thường của 13 file CSV trước khi xây dựng analytical tables.

### 1.1 `fig_products_dist.png` — Phân phối Danh mục Sản phẩm

<img src="../../reports/figures/fig_products_dist.png" alt="Phân phối giá, margin %, và margin theo category của sản phẩm" width="100%">

*Phân phối giá, margin %, và margin theo category của sản phẩm*


**Trực quan hóa cái gì:**
- Phân phối giá sản phẩm (Price distribution) — histogram
- Gross margin % theo từng sản phẩm — histogram
- Margin trung bình theo category — horizontal bar chart

**Dữ liệu đầu vào:** `products.csv` (2,412 dòng, 8 cột)

**Plan:**
1. Load `products.csv`, tính `margin_pct = (price - cogs) / price * 100`
2. 3-panel subplot: (a) Histogram price với bins=40, màu `#2d6a4f`; (b) Histogram margin_pct với bins=40, màu `#52b788`; (c) Bar chart margin trung bình theo category, màu `#95d5b2`
3. Xuất → `reports/figures/fig_products_dist.png`

**Insight:**
- Median margin ~30.6% — mức lành mạnh cho ngành bán lẻ thời trang
- Streetwear chiếm 1,320/2,412 = 54.7% danh mục → rủi ro tập trung vào một category duy nhất
- Giá phân bố rộng từ 9 VND đến 40,950 VND — AOV spread lớn, cần phân tích theo segment
- GenZ chỉ có 148 sản phẩm (6.1%) → category bị under-represented dù tiềm năng tăng trưởng cao

### 1.2 `fig_customers_dist.png` — Phân phối Tệp Khách hàng

<img src="../../reports/figures/fig_customers_dist.png" alt="Signups theo năm, nhóm tuổi, và kênh thu hút khách hàng" width="100%">

*Signups theo năm, nhóm tuổi, và kênh thu hút khách hàng*


**Trực quan hóa cái gì:**
- Signups theo năm — bar chart
- Phân phối nhóm tuổi — bar chart
- Kênh acquisition — horizontal bar chart

**Dữ liệu đầu vào:** `customers.csv` (121,930 dòng)

**Plan:**
1. Groupby `signup_date.dt.year` → đếm signups/năm
2. Thứ tự nhóm tuổi: `["18-24","25-34","35-44","45-54","55+"]`
3. Channel: value_counts, sắp xếp tăng dần để barh dễ đọc

**Insight:**
- Signups tăng đều 2012→2022 — phía cầu không sụp đổ (phát hiện quan trọng: revenue collapse không phải do thiếu khách hàng mới)
- 25-34 chiếm 29.8% — young professionals, phân khúc khách hàng cốt lõi
- organic_search là kênh acquisition lớn nhất (36,450) — traffic miễn phí chiếm ưu thế
- email_campaign = 14,674 (12.0%) — kênh chưa được khai thác hết tiềm năng

### 1.3 `fig_orders_dist.png` — Phân phối Đơn hàng & Trạng thái

<img src="../../reports/figures/fig_orders_dist.png" alt="Số đơn hàng theo năm và phân phối trạng thái đơn hàng" width="100%">

*Số đơn hàng theo năm và phân phối trạng thái đơn hàng*


**Trực quan hóa cái gì:**
- Orders theo năm — bar chart
- Phân phối trạng thái đơn hàng — horizontal bar chart

**Dữ liệu đầu vào:** `orders.csv` (646,945 dòng)

**Plan:**
1. Groupby `order_date.dt.year` → orders/năm
2. `order_status.value_counts()` → phân phối trạng thái
3. 2-panel subplot

**Insight:**
- Orders: 33K (2013) → 82K (2016) → 40K (2019) → phục hồi yếu → 37K (2022)
- Cancelled: 59,462 (9.2%) — tỉ lệ hủy đáng kể, COD là nguyên nhân chính
- Returned: 36,142 (5.6%) — tỉ lệ nhỏ nhưng wrong_size = 14K chiếm ưu thế
- Order volume đỉnh-đáy giảm -55% (82K→37K) — khớp với revenue collapse

### 1.4 `fig_sales_overview.png` — Phân tích Sâu Biến Mục tiêu (Target Variable)

<img src="../../reports/figures/fig_sales_overview.png" alt="4-panel: Revenue năm, Revenue ngày, Margin %, và Revenue theo ngày trong tuần" width="100%">

*4-panel: Revenue năm, Revenue ngày, Margin %, và Revenue theo ngày trong tuần*


**Trực quan hóa cái gì (4-panel subplot):**
- Annual Revenue (triệu VND) bar chart — highlight cú sốc 2019 màu đỏ
- Daily Revenue time series với đường rolling mean 90 ngày
- Annual Gross Margin % bar chart vs đường mean baseline
- Avg Revenue theo Ngày trong Tuần — highlight Thứ Tư > Thứ Bảy

**Dữ liệu đầu vào:** `sales.csv` (3,833 dòng, 2012-07-04 → 2022-12-31)

**Plan:**
1. Annual: groupby `year`.Revenue.sum() / 1e6
2. Daily: vẽ raw + rolling(90).mean() overlay
3. Margin: `(Revenue - COGS) / Revenue * 100` theo năm
4. DoW: `Date.dt.day_name()` groupby mean

**Insight:**
- **Đỉnh 2016: 2,105 triệu → đáy 2019: 1,137 triệu = -46% doanh thu**
- Phục hồi 2022 = 1,170 triệu, chỉ đạt 55.5% đỉnh — tổn thương cấu trúc, không phải chu kỳ
- Margin ổn định ~12-16% suốt 10 năm → vấn đề là VOLUME, không phải MARGIN
- **Thứ Tư > Thứ Bảy** về doanh thu trung bình — ngược với quy luật bán lẻ thông thường
- Đây là "giả thuyết office-worker": khách mua sắm giờ nghỉ trưa bằng mobile, không phải shopping cuối tuần

### 1.5 `fig_web_traffic.png` — Phân tích Traffic Web

<img src="../../reports/figures/fig_web_traffic.png" alt="Sessions theo ngày và sessions trung bình theo nguồn traffic" width="100%">

*Sessions theo ngày và sessions trung bình theo nguồn traffic*


**Trực quan hóa cái gì:**
- Daily Sessions (K) time series với đường rolling mean 90 ngày
- Avg daily sessions theo nguồn traffic — horizontal bar chart

**Dữ liệu đầu vào:** `web_traffic.csv` (3,652 dòng, 2013-01-01 → 2022-12-31)

**Plan:**
1. Gom daily aggregate: tổng sessions, visitors, pageviews trên tất cả traffic sources
2. Vẽ raw daily + đường MA 90d overlay
3. Source avg: `web_traffic.groupby('traffic_source').sessions.mean()`

**Insight:**
- Sessions tăng từ 6.8M (2013) → 11.1M (2022) = +63% — traffic không phải là vấn đề
- **Traffic tăng +67% nhưng Doanh thu giảm -46%** = conversion collapse, KHÔNG phải demand collapse
- organic_search là nguồn traffic lớn nhất — đầu tư SEO đang hoạt động hiệu quả
- web_traffic chỉ có đến 2022-12-31 → không cover test period → chỉ dùng lag features cho Part 3

### 1.6 `fig_inventory_paradox.png` — Nghịch lý Tồn kho

<img src="../../reports/figures/fig_inventory_paradox.png" alt="Phân tích 3-panel hiện tượng nghịch lý tồn kho 50.6%" width="100%">

*Phân tích 3-panel hiện tượng nghịch lý tồn kho 50.6%*


**Trực quan hóa cái gì (3-panel):**
- Phân phối tổ hợp flag (%) — stacked bar
- Scatter: stockout_days vs stock_on_hand, tô màu theo trạng thái nghịch lý
- Box plot: units_received theo nhóm flag — kiểm định "giả thuyết timing"

**Dữ liệu đầu vào:** `inventory.csv` (60,247 dòng, 126 tháng × 1,624 sản phẩm)

**Plan:**
1. Flag combos: đếm dòng có (stockout_flag, overstock_flag) = (1,1), (1,0), (0,1), (0,0)
2. Scatter: sample 3,000 dòng, đỏ=nghịch lý, xanh=bình thường
3. Boxplot: so sánh 4 nhóm trên units_received — Mann-Whitney U test cho ý nghĩa thống kê

**Insight:**
- **50.6% product-months CÓ CẢ stockout_flag=1 VÀ overstock_flag=1** — tưởng như không thể xảy ra
- Nghịch lý được giải thích bởi TIMING: hết hàng đầu tháng → đặt bổ sung khẩn cấp → nhận hàng cuối tháng → tồn dư
- fill_rate trung bình 96.1% nhưng stockout_flag = 67.3% — KPI illusion (1 ngày stockout trong tháng 31 ngày vẫn = 96.8% fill rate)
- units_received cao hơn đáng kể trong tháng nghịch lý (Mann-Whitney p < 0.05) — ủng hộ giả thuyết timing
- Hàm ý kinh doanh: cần chu kỳ tái đặt hàng ngắn hơn (7 ngày thay vì 30 ngày) + điều chỉnh safety stock

---
## Phần 2: EDA Khám phá — Trực quan hóa Phân tích Khám phá
**Nguồn:** `notebooks/02_exploration/01_eda_exploratory.ipynb`

**Mục tiêu:** Xác minh 6 context facts, kiểm định 7 giả thuyết, phát hiện structural breaks, chọn ra các câu chuyện cho Dashboard.

### 2.1 `fig_funnel_analysis.png` — Phân tích Phễu Toàn diện (Traffic → Orders → Revenue)

<img src="../../reports/figures/fig_funnel_analysis.png" alt="Phễu 4-panel: chỉ số indexed, conversion rate, waterfall, revenue/session" width="100%">

*Phễu 4-panel: chỉ số indexed, conversion rate, waterfall, revenue/session*


**Trực quan hóa cái gì (4-panel phễu toàn diện):**
1. **Hiệu suất Indexed (2013=100):** Đường Sessions, Orders, Revenue — biểu đồ "mâu thuẫn chính"
2. **Tỉ lệ Chuyển đổi Session→Order:** Bar chart theo năm, highlight năm dưới trung bình màu đỏ
3. **Waterfall Phễu 2016 vs 2022:** So sánh bar 5 giai đoạn phễu
4. **Doanh thu trên mỗi Session (VND):** Line chart — chỉ số hiệu quả traffic

**Dữ liệu đầu vào:** `abt_daily.parquet` — sessions_total, n_orders, n_delivered, Revenue

**Plan:**
1. Gom năm: tổng sessions, orders, delivered, revenue. Tính conversion = orders/sessions, revenue_per_session
2. Index tất cả chỉ số về baseline 2013=100
3. Vẽ các đường indexed phân kỳ sau 2016 → bằng chứng trực quan
4. Waterfall: giá trị 5 giai đoạn chuẩn hóa cho 2016 vs 2022

**Insight:**
- **Sessions +67%, Orders -55%, Revenue -46% (2013→2022 indexed)** — đây là mâu thuẫn cốt lõi
- Session-to-order conversion giảm từ ~1.0% (đỉnh) → ~0.4% (2022) — phễu bị hỏng ở khâu CONVERSION, không phải ACQUISITION
- Revenue/session giảm từ ~220 VND/session → ~100 VND — mỗi khách truy cập tạo ra ít giá trị hơn trước
- Nếu traffic 2022 (11.1M sessions) có conversion rate như 2016 → ~294 orders/ngày thay vì ~100 → +194 orders/ngày bị mất
- **Revenue collapse là thất bại về supply-side và conversion, KHÔNG phải demand destruction**

### 2.2 `fig_cohort_analysis.png` — Phân tích Cohort & Giữ chân Khách hàng

<img src="../../reports/figures/fig_cohort_analysis.png" alt="3-panel cohort: heatmap retention, retention theo kênh, LTV theo kênh" width="100%">

*3-panel cohort: heatmap retention, retention theo kênh, LTV theo kênh*


**Trực quan hóa cái gì (3-panel):**
1. **Heatmap Retention:** signup_year × months_since_signup — màu = retention %
2. **Channel Retention M3/M6/M12:** Grouped bar chart — kênh nào giữ chân tốt nhất?
3. **LTV theo Kênh:** Horizontal bar — doanh thu trung bình hàng tháng trên mỗi khách hàng active

**Dữ liệu đầu vào:** `abt_customer_cohort.parquet` — grain customer × months_since_signup

**Plan:**
1. Pivot cohort: groupby(signup_year, months_since_signup).customer_id.nunique() / cohort_size × 100
2. Lọc kênh: cô lập dòng M0, M3, M6, M12 → retention % theo kênh
3. LTV: avg revenue_in_month theo acquisition_channel (M0-M12)

**Insight:**
- Retention giảm ~50% sau M3, ~30% còn lại sau M6 — suy giảm điển hình của e-commerce
- email_campaign có LTV thấp nhất trong các kênh acquisition — nhưng cancel rate cũng thấp nhất (nghịch lý)
- organic_search & referral có retention cao nhất → nên tập trung ngân sách acquisition vào đây
- Cohort 2017-2018 có retention thấp hơn vs 2013-2016 → chất lượng cohort suy giảm — ủng hộ H1
- **Hành động:** Chiến dịch win-back M3+ cho cohort đã rời bỏ → cơ hội dễ nắm bắt nhất

### 2.3 `fig_revenue_decomposition.png` — Phân rã Doanh thu & Phát hiện Structural Break

<img src="../../reports/figures/fig_revenue_decomposition.png" alt="4-panel: phân rã revenue, AOV, structural break, items/order vs discount" width="100%">

*4-panel: phân rã revenue, AOV, structural break, items/order vs discount*


**Trực quan hóa cái gì (4-panel chẩn đoán):**
1. **Orders vs Revenue vs Price (index 2016=100):** Yếu tố nào giảm mạnh nhất?
2. **Average Order Value (AOV):** Bar chart net revenue/order theo năm
3. **Structural Break Detection:** Đường trend Pre-2019 vs Post-2019 với Chow test
4. **Items per Order vs Discount Rate:** Bar kép + line

**Dữ liệu đầu vào:** `abt_orders_enriched.parquet` — order-level net_revenue, unit_price, discount_amount

**Plan:**
1. Công thức phân rã: Revenue = n_orders × items_per_order × avg_unit_price × (1 - avg_discount_pct)
2. Index tất cả thành phần về baseline 2016
3. Hồi quy tuyến tính: fit riêng pre-2019 và post-2019 → so sánh slopes
4. Structural break xác nhận nếu slope change > 50M/năm và R² khác biệt

**Insight:**
- **Orders giảm -55% (2016→2022) nhưng AOV chỉ giảm ~5%** → ORDER VOLUME LÀ NGUYÊN NHÂN CHÍNH
- Structural break được phát hiện tại 2019: pre-trend tăng nhẹ, post-trend đi ngang/suy giảm
- avg_unit_price ổn định suốt 10 năm → xác nhận H2: vấn đề volume, không phải value
- Items per order ổn định ~4.5 → khách vẫn mua basket size tương tự
- **Kết luận: Revenue collapse thuần túy là vấn đề conversion volume — AOV không đổi, basket không đổi, ít khách hoàn tất mua hàng hơn**

### 2.4 `fig_inventory_analysis.png` — Trực quan hóa Khủng hoảng Tồn kho

<img src="../../reports/figures/fig_inventory_analysis.png" alt="4-panel inventory: heatmap stockout, lost revenue, fill rate vs stockout, flag distribution" width="100%">

*4-panel inventory: heatmap stockout, lost revenue, fill rate vs stockout, flag distribution*


**Trực quan hóa cái gì (4-panel chuyên sâu):**
1. **Stockout days heatmap:** Category × Năm — màu = avg stockout days/tháng
2. **Doanh thu Mất đi Ước tính từ Stockouts:** Bar chart theo năm
3. **Fill Rate vs Stockout Rate Scatter:** Kích thước bong bóng = sell-through rate — phát hiện KPI illusion
4. **Flag Distribution Stacked Bar:** Phân phối tổ hợp Stockout+Overstock

**Dữ liệu đầu vào:** `inventory.csv` + `abt_orders_enriched.parquet` (cho doanh thu theo category)

**Plan:**
1. Stockout heatmap: pivot_table category × year, values=stockout_days.mean()
2. Ước tính doanh thu mất: stockout_days × avg_daily_rev_per_product × n_products
3. Scatter: x=fill_rate, y=stockout_flag%, bubble_size=sell_through_rate — chú thích vùng KPI illusion
4. Flag distribution: đếm mỗi tổ hợp (stockout_flag, overstock_flag)

**Insight:**
- Doanh thu mất từ stockouts ước tính đáng kể mỗi năm
- Streetwear có stockout cao nhất nhưng cũng overstock cao nhất → inventory whipsaw
- KPI illusion: "Fill rate cao nhưng stockout cao" — fill_rate 96% không có nghĩa là không hết hàng
- sell_through_rate cao ở Casual + GenZ → đây là categories cần TĂNG phân bổ tồn kho
- **Hành động:** Giảm stockout days từ trung bình 1.16 xuống 0.5 → thu hồi ~50% doanh thu mất từ stockouts

---
## Phần 3: Dashboard Streamlit — Trực quan hóa Tương tác
**Nguồn:** `src/app/app_pages/00-08_*.py`

**Mục tiêu:** Dashboard đa trang tương tác với theme dark cinematic green, KPI cards glassmorphism, và biểu đồ Plotly. Mỗi trang bao phủ 1 "Câu chuyện" từ EDA Plan.

**Hệ thống thiết kế:** `#0A1F1A` nền tối · `#52B788` xanh lá chính · Outfit headings + Inter body + JetBrains Mono numbers

### 3.1 Trang 00 — Tổng quan (Overview Dashboard)

<img src="../../reports/figures/fig_abt_daily_overview.png" alt="Tổng quan dashboard với KPI cards và revenue trend" width="100%">

*Tổng quan dashboard với KPI cards và revenue trend*


**Trực quan hóa cái gì:**
- Revenue × Traffic dual-axis bar+line chart — biểu đồ hero
- KPI row: Tổng Doanh thu, Tổng COGS, Margin TB, Tổng Orders (glassmorphism cards)
- Bento grid: Category mix area chart, Cancel rate bar, DoW pattern, Revenue trend sparkline

**Dữ liệu đầu vào:** `abt_daily.parquet` — daily grain, tất cả metrics đã được tính sẵn

**Plan:**
1. Annual: bar cho Revenue, line (trục y phụ) cho Sessions → mâu thuẫn trực quan
2. KPI cards: `render_kpi_glass()` với label, value (fmt_vnd), delta (tăng/giảm/không đổi), caption
3. Category mix: `go.Pie` hoặc area chart — 4 categories với brand colors
4. Cancel rate: `go.Bar` — payment_method × cancel_rate

**Insight:**
- **"Doanh thu giảm -46% trong khi traffic tăng +67%"** — đây là elevator pitch, hiển thị ngay lập tức
- KPI row kể câu chuyện 30 giây: tổng doanh thu 16.4 tỷ, margin 12.5%, 647K orders
- Category mix cho thấy Streetwear chiếm ưu thế (80%) — rủi ro phụ thuộc vào một điểm duy nhất
- Cancel rate 9.2% tổng thể → COD cao nhất → đòn bẩy prescriptive có thể thấy ngay từ trang 1

### 3.2 Trang 01 — Revenue Collapse (Mô tả → Đề xuất)

<img src="../../reports/figures/fig_abt_daily_overview.png" alt="Timeline doanh thu có chú thích với waterfall decomposition" width="100%">

*Timeline doanh thu có chú thích với waterfall decomposition*


**Trực quan hóa cái gì:**
- Annotated timeline: cột doanh thu năm với chú thích đỉnh 2016 + đáy 2019
- Waterfall decomposition: ΔRevenue = Σ(sessions, conversion, AOV) contributions
- Category × Year heatmap: revenue share + YoY% theo category
- Predictive scenario slider: "Nếu conversion phục hồi về 0.65% thì sao?"

**Dữ liệu đầu vào:** `abt_daily.parquet` + `abt_orders_enriched.parquet`

**Plan:**
1. Timeline: `go.Bar` doanh thu năm, highlight 2016 xanh, 2019 đỏ, dùng `add_annotation()`
2. Waterfall: tính Δ từng thành phần giữa 2016 và 2022 → `go.Waterfall()`
3. Heatmap: pivot doanh thu theo year×category → áp dụng brand colorscale
4. Scenario: `st.slider` conversion rate → dự báo revenue = sessions × conversion_new × AOV

**Insight:**
- **2016→2019: -46% revenue = -49% orders (volume) + AOV ổn định** — waterfall định lượng attribution
- Streetwear 2018→2019: -40% YoY (80% doanh nghiệp → tác động vượt trội)
- Casual 2018→2019: +35% năm 2018 → -38% năm 2019 → mô hình bubble-pop
- Scenario: phục hồi conversion từ 0.4% → 0.65% = +55% doanh thu (~+600 triệu VND/năm)
- **Đây là đòn bẩy prescriptive mạnh nhất — ROI cao hơn chi tiêu acquisition**

### 3.3 Trang 02 — Phễu & Khách hàng (Chẩn đoán → Đề xuất)

<img src="../../reports/figures/fig_cohort_retention.png" alt="Phễu 4 giai đoạn và cohort retention heatmap" width="100%">

*Phễu 4 giai đoạn và cohort retention heatmap*


**Trực quan hóa cái gì:**
- Session → Order → Delivered → Review funnel (phễu 4 giai đoạn `go.Funnel`)
- Cohort retention heatmap: signup quarter × quarters since signup
- Channel LTV bubble chart: x=LTV/khách, y=cancel_rate, bubble size=số lượng khách
- Return reasons stacked bar theo category
- Prescriptive action cards: Sửa Size Guide, COD→Prepay, Cohort Win-Back

**Dữ liệu đầu vào:** Tất cả 5 ABT qua data_loader

**Plan:**
1. Funnel: `go.Funnel()` với 4 stages, brand colors, textinfo="value+percent previous"
2. Cohort heatmap: orders merge customers → cohort = signup_quarter, order_quarter → pivot retention
3. Bubble: `go.Scatter(mode='markers+text')` — size ∝ √customers
4. Return reasons: `go.Bar(x=category, y=count, color=return_reason)` stacked

**Insight:**
- Phễu: 25M sessions → 517K delivered = 2.1% session→delivered conversion
- **email_campaign có cancel rate cao nhất trong các kênh** — nghịch lý: acquisition rẻ nhưng rò rỉ
- COD cancel rate cao hơn 2-3× non-COD → hành động: chuyển 20% COD sang trả trước → tăng ~X triệu/năm
- wrong_size = 7,626 returns (lý do #1) → sửa size guide rẻ + ROI cao
- Cohort retention cliff tại Q3 → chiến dịch win-back nhắm khách Q3+ đã rời bỏ

### 3.4 Trang 03 — Nghịch lý Tồn kho (Chẩn đoán)

<img src="../../reports/figures/fig_inventory_analysis.png" alt="Scatter stockout vs days_of_supply và sell-through violin" width="100%">

*Scatter stockout vs days_of_supply và sell-through violin*


**Trực quan hóa cái gì:**
- Scatter: stockout_days vs days_of_supply, tô màu theo category — highlight vùng nghịch lý
- Stockout rate trend line theo thời gian
- Doanh thu mất đi ước tính bar theo category
- Sell-through rate violin distribution theo category
- KPI illusion card: "Fill rate 96% nhưng Stockout 67%"

**Dữ liệu đầu vào:** `inventory.csv` + `abt_daily.parquet`

**Plan:**
1. Scatter: `go.Scatter()` x=stockout_days, y=days_of_supply, color=category
2. Vùng nghịch lý: chú thích điểm stockout_days > 0 VÀ days_of_supply > median
3. Violin: `go.Violin()` theo category — hiển thị hình dạng phân phối sell_through_rate
4. KPI card: HTML tùy chỉnh với border-left accent color

**Insight:**
- "96% fill rate, 67% stockout — tại sao?" → 1 ngày stockout trong tháng 31 ngày = fill_rate 96.8%, nhưng flag=1
- days_of_supply mean = 912.7 ngày (2.5 năm!) — nhưng median = 240 ngày (vấn đề skewness)
- Casual có sell-through thấp nhất → ứng viên cho chiến lược markdown/clearance
- GenZ có sell-through cao nhất → category xứng đáng được phân bổ tồn kho nhiều hơn
- **Bảng lộ trình hành động**: Q1 sửa reorder triggers, Q2 tối ưu safety stock, Q3 tái cân bằng category

### 3.5 Trang 04 — Recovery Simulator (Dự báo → Đề xuất)

**Trực quan hóa cái gì:**
- 5 thanh trượt tương tác: Conversion rate, Stockout reduction, Cancel reduction, AOV boost, Session growth
- Biểu đồ dự báo trực tiếp: baseline vs optimized revenue line
- Ma trận ưu tiên Effort × Impact: scatter 2×2 với các hành động được gắn nhãn
- Lộ trình Gantt 2023: timeline triển khai Q1-Q4

**Dữ liệu đầu vào:** Dẫn xuất từ `abt_daily.parquet` với baseline metrics 2022

**Plan:**
1. Sliders: `st.slider()` cho từng đòn bẩy, mặc định = giá trị hiện tại, phạm vi ±30%
2. Projection: tính projected_revenue = baseline × (1 + ΣΔlevers), vẽ cả hai đường
3. Priority matrix: tính effort_score (1-5) + impact_score (doanh thu dự kiến tăng) → scatter 4 góc
4. Gantt: HTML bars với vị trí Q1-Q4

**Insight:**
- **Conversion lever có tỉ lệ impact-to-effort cao nhất** — sửa conversion từ 0.42% → 0.65% = +300 triệu/năm
- Stockout reduction tác động trung bình, nỗ lực trung bình — cần viết lại hệ thống tồn kho
- Cancel reduction từ 9.2% → 6% tiết kiệm ~200 triệu/năm doanh thu mất
- Session growth ít tác động nhất — xác nhận "đừng chi tiền cho acquisition, hãy sửa phễu"
- Priority matrix: Conversion + Cancel reduction trong góc high-impact/low-effort → LÀM TRƯỚC

### 3.6 Trang 05 — Patterns & Timing (Mô tả → Chẩn đoán)

<img src="../../reports/figures/eda_seasonality.png" alt="Mùa vụ theo tháng với error bars và heatmap year×month" width="100%">

*Mùa vụ theo tháng với error bars và heatmap year×month*


**Trực quan hóa cái gì:**
- Monthly seasonality bar chart với error bars — highlight đỉnh Apr-Jun, đáy Nov-Jan
- Year × Month revenue heatmap — tính nhất quán của đỉnh qua tất cả các năm
- Day-of-Week bar chart: Thứ Tư > Thứ Bảy — hiện tượng ngược trực giác
- KPI cards: Thứ Tư Hạng #1, Hạng Thứ Bảy, Mức chênh Wed vs Sat %

**Dữ liệu đầu vào:** `abt_daily.parquet`

**Plan:**
1. Monthly: groupby month → avg Revenue + std → `go.Bar()` với error_y, tô Apr-Jun xanh, Nov-Jan đỏ
2. Heatmap: pivot year×month → `go.Heatmap()` với brand colorscale, texttemplate cho giá trị
3. DoW: groupby day_name → `go.Bar()` sắp xếp Thứ Hai→Chủ Nhật, highlight Thứ Tư xanh

**Insight:**
- **Mùa cao điểm = Apr-Jun (không phải Tết)** — ngược trực giác cho bán lẻ VN (thường Nov-Feb peak)
- Dị thường Anti-Tết: Dec-Jan là tháng THẤP NHẤT dù là mùa mua sắm truyền thống VN
- **Mô hình nhất quán trên TẤT CẢ 10 năm** → không phải fluke một năm, là chu kỳ mùa thực sự
- Doanh thu Thứ Tư CAO HƠN Thứ Bảy — "Giả thuyết office-worker": mua sắm giờ nghỉ trưa bằng mobile
- **Hàm ý cho dự báo:** Bất kỳ mô hình nào dùng đặc trưng ngày lễ VN chung (tăng Tết) sẽ DỰ BÁO QUÁ CAO Q1 và DỰ BÁO QUÁ THẤP Q2

### 3.7 Trang 06 — Promo ROI (Chẩn đoán)

**Trực quan hóa cái gì:**
- Discount value distribution: grouped bar theo promo_type × discount_value
- Promo calendar Gantt chart: tất cả 50 promo trên timeline — hiển thị nhịp 6-4-6-4
- Revenue lift bars: avg daily revenue theo promo status (Không promo, Percentage, Fixed, Cả hai)
- Lift % vs no-promo baseline: bar chart với màu tích cực/tiêu cực
- KPI cards: avg revenue/ngày theo promo_type, số ngày, lift %
- Targeting gap donut: 80% promos có applicable_category = null

**Dữ liệu đầu vào:** `promotions.csv` + `abt_daily.parquet`

**Plan:**
1. Discount distribution: `go.Bar(barmode='group')` — màu theo promo_type (percentage=xanh, fixed=vàng)
2. Gantt: horizontal bar mỗi promo, x=start_date, width=duration_days, color=promo_type
3. Lift: abt_train flag promo days → groupby promo_type_label → avg Revenue → bar
4. Lift %: (avg_rev / no_promo_avg - 1) × 100 → xanh nếu dương, đỏ nếu âm
5. Targeting donut: `go.Pie(hole=0.55)` — 80% null vs 20% targeted

**Insight:**
- **Mẫu calendar 6-4-6-4 đều bất thường** — thiết kế dataset tổng hợp, không phải lập kế hoạch promo thực tế
- Chỉ có 2 mức discount thực sự có ý nghĩa: 10-20% (percentage) và 50 VND (fixed) — fixed discount không đáng kể
- Percentage promos tạo lift đo lường được vs baseline không promo
- Fixed promos (50 VND trên items giá 1,000-100,000 VND) = 0.05%-5% discount hiệu quả — vô nghĩa về kinh tế
- 80% promos không nhắm category cụ thể → xem promo như tín hiệu traffic, không phải công cụ quản lý margin
- **promo_id_2 (stackable) chỉ xuất hiện trong 0.03% order items** — stackability hầu như không dùng đến

### 3.8 Trang 07 — Phân tích Địa lý (Mô tả)

**Trực quan hóa cái gì:**
- Top-20 cities doanh thu bar chart
- Top-3 city KPI cards: doanh thu từng thành phố, avg order value, thị phần %
- Acquisition channel × city stacked bar
- AOV bubble plot: city_avg_AOV vs city_total_revenue

**Dữ liệu đầu vào:** `abt_orders_enriched.parquet` (merge với geography)

**Plan:**
1. City revenue: groupby city → sum revenue → top-20 → `go.Bar(orientation='h')`
2. Channel×city: cross-tab → `go.Bar(stacked)` với CHANNEL_COLORS
3. Bubble: `go.Scatter(mode='markers')` — size=revenue, x=AOV, y=city, text=city

**Insight:**
- East region chiếm ưu thế 7.29 tỷ VND (Q7 đã xác minh) — ~47% tổng doanh thu
- Top-3 cities có thể là TP.HCM, Hà Nội, Đà Nẵng — phân phối e-commerce VN điển hình
- paid_search over-indexed ở thành thị, organic_search mạnh hơn ở nông thôn — tương tác channel×geo
- AOV cao hơn ở East region (thành thị) vs Central/West — khách thành thị chi tiêu nhiều hơn mỗi đơn
- Thị phần doanh thu theo vùng ổn định theo thời gian (H6: chi-squared không ý nghĩa) → không có structural break địa lý

### 3.9 Trang 08 — Cohort & Kiểm định Giả thuyết (Chẩn đoán)

<img src="../../reports/figures/fig_cohort_analysis.png" alt="Đường cong retention cohort theo kênh và RFM segmentation" width="100%">

*Đường cong retention cohort theo kênh và RFM segmentation*


**Trực quan hóa cái gì:**
- Cohort retention curves theo kênh acquisition — line chart M0→M24
- LTV theo kênh M12 lũy kế — bar chart
- RFM segmentation donut: Champions/Loyal/At-Risk/Lost
- 7 hypothesis test p-value bar chart (đường α=0.05)
- AOV by year boxplot + Conversion trend line
- Bảng tóm tắt kết quả: ID, giả thuyết, test, statistic, p-value, kết luận

**Dữ liệu đầu vào:** `abt_customer_cohort.parquet`, `abt_orders_enriched.parquet`, `inventory.csv`

**Plan:**
1. Cohort retention: groupby acquisition_channel, months_since_signup → retention% → go.Scatter
2. LTV: doanh thu lũy kế M0→M12 mỗi kênh → go.Scatter cumulative line
3. RFM: recency (ngày từ lần mua cuối), frequency (số đơn), monetary (tổng chi) → phân đoạn quartile → go.Pie
4. Hypothesis p-values: tính từng test → go.Bar với p-values, α=0.05 hline
5. Scipy stats: mannwhitneyu, kruskal, spearmanr, linregress, fisher_exact

**Insight:**
- email_campaign có LTV thấp nhất dù là kênh acquisition lớn thứ 2
- organic_search + referral tạo ra khách hàng LTV cao nhất → nên đầu tư vào đây
- RFM: Champions (~25%) tạo ra ~70% doanh thu → chương trình nurture rất quan trọng
- **H2 ĐƯỢC XÁC NHẬN:** AOV ổn định qua các năm (Kruskal-Wallis p>0.05) → volume collapse confirmed
- **H4 ĐƯỢC XÁC NHẬN:** Conversion đang giảm (linear regression p<0.05, slope âm) → phễu đang rò rỉ
- **H7 ĐƯỢC XÁC NHẬN:** COD cancel rate cao hơn đáng kể (Fisher's exact p<0.001) → đòn bẩy vận hành
- At-Risk segment (25%) = mục tiêu tái kích hoạt ROI cao nhất: họ từng mua, chỉ là không gần đây

---
## Phần 4: Dự báo & SHAP — Trực quan hóa Mô hình Dự báo
**Nguồn:** `notebooks/06_output/part3_pipeline.ipynb`

**Mục tiêu:** Dự báo Revenue + COGS cho giai đoạn test 2023-01-01 → 2024-07-01 (548 ngày) bằng LightGBM ensemble + seasonal baseline. SHAP để giải thích mô hình.

### 4.1 Chuỗi A1-A9 — EDA Khám phá cho Dự báo

<img src="../../reports/figures/A1_sales_overview.png" alt="A1: Tổng quan Doanh thu + COGS + Gross Margin" width="100%">

*A1: Tổng quan Doanh thu + COGS + Gross Margin*


**Trực quan hóa cái gì (9 biểu đồ):**

| Hình | Nội dung | Loại Biểu đồ |
|---|---|---|
| **A1** `A1_sales_overview.png` | Revenue + COGS + Gross Margin % (3-panel time series) | Area + line |
| **A2** `A2_revenue_cogs.png` | Revenue vs COGS scatter (tô năm) + phân phối + tỉ lệ COGS/revenue | Scatter + histogram + line |
| **A3** `A3_seasonality.png` | Monthly seasonality bars + DOW bars + Month×DOW heatmap | Bar + heatmap |
| **A4** `A4_annual_trend.png` | Annual revenue bars + YoY% growth + Revenue=COGS+GrossProfit stacked | Bar + stacked area |
| **A5** `A5_decomposition.png` | Trend(365d MA) + Seasonal Index + Residual | 3-panel time series |
| **A6** `A6_promotions.png` | Revenue theo promo count + Revenue vs Active Promos timeline + promo type pie | Bar + dual-axis + pie |
| **A7** `A7_web_traffic.png` | Lag correlation (Revenue vs Sessions[t-lag]) + scatter + bounce theo nguồn + sessions theo nguồn | Bar + scatter |
| **A8** `A8_inventory.png` | Fill Rate vs Revenue scatter + Stockout trend + Sell-through theo category + Stock on hand | Scatter + line + bar |
| **A9** `A9_orders_returns.png` | Order count vs Revenue scatter + order status breakdown + returns timeline + payment method AOV | Scatter + bar + line |



<img src="../../reports/figures/A2_revenue_cogs.png" alt="A2: Revenue vs COGS — Mối quan hệ" width="100%">

*A2: Revenue vs COGS — Mối quan hệ*


<img src="../../reports/figures/A3_seasonality.png" alt="A3: Mùa vụ — Monthly, DOW, Month×DOW" width="100%">

*A3: Mùa vụ — Monthly, DOW, Month×DOW*


<img src="../../reports/figures/A4_annual_trend.png" alt="A4: Xu hướng năm — Revenue, YoY%, Stacked Revenue" width="100%">

*A4: Xu hướng năm — Revenue, YoY%, Stacked Revenue*


<img src="../../reports/figures/A5_decomposition.png" alt="A5: Phân rã tín hiệu — Trend, Seasonal, Residual" width="100%">

*A5: Phân rã tín hiệu — Trend, Seasonal, Residual*


<img src="../../reports/figures/A6_promotions.png" alt="A6: Promotions — Revenue theo promo, timeline, pie" width="100%">

*A6: Promotions — Revenue theo promo, timeline, pie*


<img src="../../reports/figures/A7_web_traffic.png" alt="A7: Web Traffic — Lag correlation, scatter, bounce" width="100%">

*A7: Web Traffic — Lag correlation, scatter, bounce*


<img src="../../reports/figures/A8_inventory.png" alt="A8: Inventory — Fill Rate, Stockout trend, Sell-through" width="100%">

*A8: Inventory — Fill Rate, Stockout trend, Sell-through*


<img src="../../reports/figures/A9_orders_returns.png" alt="A9: Orders & Returns — Order count, status, returns, payment AOV" width="100%">

*A9: Orders & Returns — Order count, status, returns, payment AOV*


**Dữ liệu đầu vào:** Tất cả 13 CSV được load raw → aggregated cho visualization

**Plan:**
Mỗi A-series figure được thiết kế để trả lời 1 câu hỏi forecasting cụ thể:
- A1: Hình dạng biến mục tiêu → phát hiện trend, volatility, outliers
- A2: Mối quan hệ Revenue-COGS → dự báo COGS cùng với Revenue (Pearson r)
- A3: Mùa vụ lịch → cơ sở cho cyclic features (sin/cos encoding)
- A4: Xu hướng năm → cơ sở cho growth rate features
- A5: Phân rã tín hiệu → cơ sở cho cấu trúc mô hình trend+seasonal+residual
- A6: Hiệu ứng promo → cơ sở cho promo_count, promo_discount_sum features
- A7: Web traffic lag → cơ sở cho lag-1 web features (phân tích lag tốt nhất)
- A8: Tín hiệu tồn kho → cơ sở cho inv_fill_rate, inv_stockout features
- A9: Tín hiệu đơn hàng → cơ sở cho ord_count_lag features

**Insight (tổng hợp forecasting EDA):**
- Revenue-COGS Pearson r ≈ 0.99 → dự báo cả hai cùng lúc khả thi
- Best lag cho sessions→revenue = 1 ngày → đưa wt_sessions lag-1 làm feature
- Seasonal CV ~15-20% → mô hình mùa mạnh → thêm sin/cos calendar features
- Promo uplift ~10-15% trung bình → đáng kể → đưa promo features
- Fill_rate-Revenue correlation vừa phải (r ~0.3) → đưa vào nhưng trọng số thấp
- Order count có correlation mạnh nhất với Revenue (r ~0.8) → lag-365 là feature quan trọng

### 4.2 `D_cv_folds.png` — Kết quả Cross-Validation theo Fold

<img src="../../reports/figures/D_cv_folds.png" alt="5-fold TimeSeriesSplit: actual vs predicted + MAE bar chart" width="100%">

*5-fold TimeSeriesSplit: actual vs predicted + MAE bar chart*


**Trực quan hóa cái gì:**
- 5-fold TimeSeriesSplit actual vs predicted lines
- Summary MAE bar chart theo fold

**Dữ liệu đầu vào:** X_train (3,468 dòng, 218 features), y_rev (Revenue target)

**Plan:**
1. TimeSeriesSplit(n_splits=5, gap=14d) — expanding window, gap 14 ngày chống leakage
2. Ensemble 3 mô hình: LGB-MAE (65%) + LGB-Huber (20%) + Seasonal baseline (15%)
3. Mỗi fold: vẽ actual (xanh) vs LGB-MAE (cam đứt) vs Blend (xanh lá chấm) cho 300 điểm mẫu
4. Summary bar: mean ensemble MAE theo fold

**Insight:**
- Ensemble MAE = 950K ± 162K (R²=0.69) — chấp nhận được cho dữ liệu doanh thu ngày nhiễu
- LGB-MAE solo: MAE=759K (R²=0.78) — thực sự LGB-MAE vượt trội hơn Huber riêng
- Huber model hoạt động kém (MAE=2.2M, R²=-0.53) — sai loss function cho dữ liệu này
- Seasonal baseline thêm ổn định ở biên test (Jan 2023, Jul 2024) nơi lag features mỏng
- Fold stability: MAE range 700K→1.2M, spread ~500K nhất quán qua các fold
- **Fold tốt nhất (MAE=700K) dùng nhiều dữ liệu training nhất → lịch sử dài hơn giúp ích đáng kể**

### 4.3 `E_shap_final.png` — SHAP Feature Importance

<img src="../../reports/figures/E_shap_final.png" alt="SHAP beeswarm: top-25 features + SHAP mean |importance| bar chart" width="100%">

*SHAP beeswarm: top-25 features + SHAP mean |importance| bar chart*


**Trực quan hóa cái gì:**
- SHAP beeswarm plot: top-25 features, màu=giá trị feature (đỏ=cao, xanh=thấp)
- SHAP mean |importance| bar chart: top-25 features xếp hạng

**Dữ liệu đầu vào:** Mô hình LGB-MAE đã train + 800 mẫu SHAP explainer

**Plan:**
1. `shap.TreeExplainer(rev_mae_m)` trên 800 mẫu training ngẫu nhiên
2. `shap.summary_plot(shap_values, X_shap, max_display=25)` → beeswarm
3. Mean |SHAP|: `pd.DataFrame({'feature': cols, 'shap_mean': abs(shap_values).mean(0)}).sort_values()`

**Insight (Top Revenue Drivers theo SHAP):**

| Hạng | Feature | Ý nghĩa Kinh doanh | Hướng |
|---|---|---|---|
| 1 | `Revenue_l1` | Doanh thu hôm qua → momentum ngắn hạn | ↑ gần đây → ↑ dự báo |
| 2 | `COGS_l1` | COGS hôm qua (tương quan với Revenue) | ↑ COGS → ↑ Revenue |
| 3 | `Revenue_rmin7` | Min revenue 7 ngày qua → chỉ báo "sàn" | Sàn cao hơn → dự báo cao hơn |
| 4 | `COGS_l364` | Neo mùa: COGS cùng ngày năm trước | Mẫu lặp hàng năm |
| 5 | `Revenue_2yoy_mean` | Trung bình 2 năm trước → baseline dài hạn | Ổn định hóa dự đoán |
| 6 | `Revenue_l7` | Revenue 1 tuần trước → mẫu tuần | Mùa vụ tuần mạnh |
| 7 | `ord_count_l365` | Order volume năm trước → proxy cầu | ↑ orders lịch sử → ↑ dự báo |
| 8 | `cal_day` | Ngày trong tháng (1-31) | Đầu tháng thấp, cuối tháng cao |
| 9 | `promo_discount_sum` | Tổng discount promos đang active | ↑ discount → ↑ Revenue |
| 10 | `cal_doy_sin/cos` | Sin/Cos day-of-year → chu kỳ năm mượt | Nắm bắt đỉnh Apr-Jun |

**Kết luận chính:** SHAP xác nhận **temporal features chiếm ưu thế** (lags, rolling means, seasonal) — external features (promo, web, inventory) là thứ yếu nhưng vẫn có ý nghĩa. Mô hình học nhịp mùa + momentum ngắn hạn.

### 4.4 `G_forecast_final.png` — Dự báo Cuối cùng

<img src="../../reports/figures/G_forecast_final.png" alt="2-panel: 2021-2022 in-sample fit + 2023-2024 forecast" width="100%">

*2-panel: 2021-2022 in-sample fit + 2023-2024 forecast*


**Trực quan hóa cái gì (2-panel):**
- 2021-2022 in-sample fit: actual vs fitted (LGB ensemble)
- 2023-2024 forecast: đường Revenue + COGS + Gross Profit

**Dữ liệu đầu vào:** submission.csv (548 dòng) + X_train fitted values

**Plan:**
1. In-sample: dự đoán X_train → kiểm tra MAE → vẽ zoom 2021-2022
2. Forecast: dự đoán từng ngày iterative (548 ngày test) → vẽ 3 đường (Revenue, COGS, Gross Profit)
3. Clip dự đoán âm về 0

**Insight:**
- In-sample MAE ~855K, R²=0.75 → mô hình nắm bắt ~75% phương sai doanh thu trên training data
- Dự báo cho thấy mẫu mùa như kỳ vọng: đỉnh Apr-Jun, đáy Nov-Jan (nhất quán với training)
- Revenue forecast trung bình ~3.5 triệu/ngày — hơi thấp hơn trung bình 2022 (~3.2 triệu) → mô hình kỳ vọng tiếp tục suy giảm chậm
- COGS bám sát Revenue (như kỳ vọng, r=0.99 trong training)
- Gross Profit dương mỗi ngày → không có dự đoán margin âm (clip hoạt động)
- **Hạn chế:** Mô hình không thể dự đoán structural shifts — nếu doanh nghiệp thay đổi vận hành (sửa conversion, giảm stockout), dự báo sẽ underestimate

---
## Phần 5: Bảng Tham chiếu Chéo — Chart Nào Trả lời Câu hỏi Kinh doanh Nào

| Câu hỏi Kinh doanh | Biểu đồ Chính | Biểu đồ Phụ | Vị trí |
|---|---|---|---|
| **Q: Revenue collapse nguyên nhân?** | fig_funnel_analysis (indexed divergence) | fig_revenue_decomposition (structural break) | `01_eda_exploratory.ipynb` + Streamlit Trang 01 |
| **Q: Traffic tăng, revenue giảm — tại sao?** | fig_funnel_analysis (conversion bar chart) | Streamlit Trang 00 hero chart | `01_eda_exploratory.ipynb` + Streamlit Trang 00 |
| **Q: Conversion problem hay traffic problem?** | fig_funnel_analysis (revenue/session line) | Streamlit Trang 02 funnel | `01_eda_exploratory.ipynb` + Streamlit Trang 02 |
| **Q: Volume hay AOV giảm?** | fig_revenue_decomposition (orders vs revenue vs price indexed) | AOV bar chart theo năm | `01_eda_exploratory.ipynb` |
| **Q: Category nào bị ảnh hưởng nặng nhất?** | Category×Year heatmap (Streamlit Trang 01) | YoY% change bar theo category | Streamlit Trang 01 |
| **Q: Inventory có vấn đề không?** | fig_inventory_paradox (flag distribution) | fig_inventory_analysis (fill rate vs stockout scatter) | `00_data_profiling.ipynb` + Streamlit Trang 03 |
| **Q: Mùa cao điểm thực sự là khi nào?** | Monthly seasonality bars (Streamlit Trang 05) | Year×Month heatmap | Streamlit Trang 05 + `part3_pipeline.ipynb` A3 |
| **Q: Wednesday > Saturday — tại sao?** | DoW bar chart (Streamlit Trang 05) | fig_sales_overview DoW panel | Streamlit Trang 05 + `00_data_profiling.ipynb` |
| **Q: Promo có hiệu quả không?** | Revenue lift bars (Streamlit Trang 06) | Promo Gantt calendar | Streamlit Trang 06 |
| **Q: Nên đầu tư kênh acquisition nào?** | Channel LTV bubble (Streamlit Trang 02) | fig_cohort_analysis (LTV bar) | Streamlit Trang 02 + `01_eda_exploratory.ipynb` |
| **Q: Cohort quality đang giảm?** | Cohort retention heatmap (Streamlit Trang 02) | fig_cohort_analysis heatmap | Streamlit Trang 02 + `01_eda_exploratory.ipynb` |
| **Q: COD có vấn đề cancel cao?** | Cancel rate by payment method (Streamlit Trang 00) | H7 Fisher's exact test (Streamlit Trang 08) | Streamlit Trang 00, 08 |
| **Q: Regional shift over time?** | Revenue share % line by region (H6 chart) | Top-20 cities bar (Streamlit Trang 07) | `01_eda_exploratory.ipynb` + Streamlit Trang 07 |
| **Q: Model dùng features nào quan trọng nhất?** | SHAP beeswarm (`E_shap_final.png`) | SHAP bar chart (`E_shap_final.png`) | `part3_pipeline.ipynb` |
| **Q: Forecast 2023-2024 trông như thế nào?** | `G_forecast_final.png` (đường Revenue+COGS+GP) | `D_cv_folds.png` (cross-validation) | `part3_pipeline.ipynb` |
| **Q: Nên hành động thế nào để phục hồi?** | Recovery Simulator (Streamlit Trang 04) | Prescriptive action cards (Streamlit Trang 02) | Streamlit Trang 04, 02 |

---
## Phần 6: Data Storytelling — Từ Dữ liệu đến Câu chuyện Kinh doanh

> **Nguyên tắc cốt lõi:** Mỗi biểu đồ là một đoạn văn. Mỗi insight trả lời câu hỏi **"So What?"** — Điều này có ý nghĩa gì? Doanh nghiệp nên làm gì tiếp theo?

**Mục tiêu của phần này:** Kết nối tất cả insight từ Phần 1–5 thành một **câu chuyện mạch lạc** có mở đầu, thân bài, và kết luận — giống như một bài trình bày trước ban lãnh đạo.

### 6.1 Cấu trúc Câu chuyện (Narrative Arc)

| Hồi | Câu hỏi Dẫn dắt | Insight Chính | Chart Mấu chốt |
|---|---|---|---|
| **Mở đầu — The Hook** | Doanh nghiệp đang gặp vấn đề gì? | Revenue -46% trong khi Traffic +67% — mâu thuẫn không thể bỏ qua | `fig_funnel_analysis.png` (indexed divergence) |
| **Hồi 1 — Điều tra Hiện trường** | Chuyện gì đã xảy ra, khi nào? | Revenue sụp đổ 2016→2019, phục hồi yếu 2020-2022, chưa về đỉnh | `fig_sales_overview.png` (annual bars) |
| **Hồi 2 — Chẩn đoán Nguyên nhân** | TẠI SAO revenue giảm dù traffic tăng? | Conversion collapse (-60%), KHÔNG phải demand destruction. Volume problem, not value problem. | `fig_revenue_decomposition.png` (structural break) |
| **Hồi 3 — Bằng chứng Bổ trợ** | Còn manh mối nào khác không? | Inventory paradox (50.6%), seasonal anomaly (Apr-Jun peak), Wed>Sat, promo synthetic | `fig_inventory_paradox.png` + Streamlit Trang 05 |
| **Hồi 4 — Dự báo Tương lai** | Nếu không làm gì, điều gì sẽ xảy ra? | Baseline forecast: tiếp tục suy giảm chậm, Revenue ~3.5M/ngày | `G_forecast_final.png` |
| **Cao trào — "So What?"** | Vậy thì SAO? Doanh nghiệp nên LÀM GÌ? | 5 đòn bẩy phục hồi, Conversion #1 ROI, COD→Prepay #2 | Recovery Simulator (Streamlit Trang 04) |
| **Kết — Lộ trình Hành động** | Bắt đầu từ đâu? Bao giờ? | Q1: Size guide + Win-back. Q2: COD→Prepay. Q3: Inventory. Q4: Category rebalance. | Gantt roadmap (Streamlit Trang 04) |

### 6.2 Mở đầu — The Hook: "Doanh thu giảm một nửa, nhưng khách hàng vẫn đến"

**Câu chuyện bắt đầu với một con số gây sốc.**

Năm 2016, doanh nghiệp đạt đỉnh 2.1 tỷ VND/năm. Ba năm sau, 2019, doanh thu chỉ còn 1.1 tỷ — **giảm 46%**. Nhưng đồng thời, lượng truy cập web lại **tăng 67%** (6.8M → 11.1M sessions). Khách hàng mới đăng ký **tăng đều mỗi năm**.

Đây là mâu thuẫn không thể bỏ qua: **Nhiều người đến hơn, nhưng doanh thu ngày càng ít đi.**

| Insight | So What? |
|---|---|
| Revenue -46% (2016→2019) | Đây không phải suy thoái nhẹ — đây là khủng hoảng cấu trúc. Cần hành động ngay, không thể chờ "tự phục hồi". |
| Traffic +67% cùng kỳ | Vấn đề KHÔNG nằm ở marketing hay acquisition. Tăng ngân sách quảng cáo sẽ KHÔNG giải quyết được gốc rễ. |
| Signups tăng đều 2012-2022 | Phía cầu khỏe mạnh. Khách hàng vẫn muốn mua. Họ chỉ KHÔNG THỂ hoặc KHÔNG MUỐN hoàn tất giao dịch. |
| Margin ổn định 12-16% | Sản phẩm không bị mất giá. Đây là vấn đề VOLUME — doanh nghiệp bán được ÍT hơn, chứ không phải bán RẺ hơn. |

> **🎯 "So What?" Tổng:** Đừng chi tiền vào marketing. Đừng giảm giá. Hãy tìm ra TẠI SAO khách không hoàn tất mua hàng.

### 6.3 Hồi 1 & 2 — Điều tra & Chẩn đoán: "Phễu bị thủng, không phải cạn nguồn"

**Sau khi xác định vấn đề, câu hỏi tiếp theo là: TẠI SAO?**

Phân tích phễu (funnel analysis) cho thấy câu trả lời rõ ràng: **Session-to-Order conversion giảm từ ~1.0% xuống ~0.4%** — tức là cứ 1,000 người ghé thăm, trước đây có 10 người mua, nay chỉ còn 4.

Phân rã doanh thu (revenue decomposition) xác nhận: **số đơn hàng giảm 55%** nhưng **giá trị trung bình mỗi đơn (AOV) gần như không đổi**. Cùng một sản phẩm, cùng mức giá, nhưng ít người nhấn "Đặt mua" hơn hẳn.

<img src="../../reports/figures/fig_funnel_analysis.png" alt="Funnel analysis — Sessions vs Orders vs Revenue indexed" width="100%">

*Funnel Analysis: Đường Sessions tăng, Orders & Revenue giảm rõ rệt sau 2016*

| Insight | So What? |
|---|---|
| Conversion giảm 60% (1.0%→0.4%) | **Đây là vấn đề TRANG WEB/APP, không phải vấn đề SẢN PHẨM.** UX checkout, tốc độ tải trang, trust signals, payment options — đây là những thứ cần audit. |
| AOV ổn định ~200K VND | Không cần thay đổi chiến lược giá. Không cần upsell. Chỉ cần khách hoàn tất giao dịch họ ĐÃ ĐỊNH thực hiện. |
| Items/order ổn định ~4.5 | Kích thước giỏ hàng không đổi. Khách vẫn muốn mua cùng một lượng hàng. Vấn đề nằm ở bước CUỐI CÙNG của phễu. |
| Structural break tại 2019 | Có một SỰ KIỆN cụ thể làm gãy conversion, không phải xu hướng từ từ. Cần tìm ra sự kiện đó (thay đổi nền tảng? đối thủ mới? khủng hoảng truyền thông?). |
| email_campaign LTV thấp + cancel rate cao | Kênh acquisition rẻ nhất nhưng chất lượng khách kém nhất. ROI thực sự của email campaign có thể ÂM nếu tính cả chi phí xử lý đơn hủy. |

> **🎯 "So What?" Tổng:** Audit UX checkout flow. Tìm sự kiện 2019. Đánh giá lại ROI thực của email campaign (tính cả cancel rate, không chỉ acquisition cost).

### 6.4 Hồi 3 — Bằng chứng Bổ trợ: "Không chỉ conversion — cả hệ thống đang có vấn đề"

**Conversion là nguyên nhân chính, nhưng không phải duy nhất.** Ba manh mối khác từ dữ liệu vẽ nên bức tranh toàn cảnh hơn:

#### A. Nghịch lý Tồn kho (Inventory Paradox)

**50.6% sản phẩm-tháng CÓ CẢ hết hàng VÀ tồn dư.** Nghe như không thể, nhưng lại là thực tế. Cơ chế: hết hàng đầu tháng → đặt bổ sung gấp → nhận hàng cuối tháng → tồn kho dư thừa.

| Insight | So What? |
|---|---|
| 50.6% product-months paradox | Hệ thống tái đặt hàng đang hoạt động SAI NHỊP. Không phải thiếu data, mà là sai chu kỳ — cần chuyển từ chu kỳ 30 ngày xuống 7-14 ngày. |
| Fill rate 96% nhưng stockout 67% | ĐỪNG dùng fill rate làm KPI chính. Nó che giấu vấn đề. Dùng "số ngày stockout/tháng" và "lost revenue từ stockout" làm KPI thay thế. |
| Streetwear: cao nhất cả stockout lẫn overstock | Category chiếm 80% doanh thu đang bị quản lý tồn kho kém nhất. Đây là ưu tiên SỐ 1 để tối ưu. |

#### B. Mùa vụ Ngược (Apr-Jun Peak)

Đỉnh doanh thu là **tháng 4-6, KHÔNG phải Tết (tháng 11-2)** — hoàn toàn ngược với bán lẻ Việt Nam truyền thống.

| Insight | So What? |
|---|---|
| Đỉnh Apr-Jun, đáy Nov-Jan | KHÔNG dùng calendar mặc định của Việt Nam (Tết, Noel) để lập kế hoạch marketing và tồn kho. Dùng lịch RIÊNG của doanh nghiệp này. |
| Nhất quán 10 năm | Đây là đặc trưng thực sự của tập khách hàng, không phải nhiễu. Mọi mô hình dự báo, lập ngân sách, và chiến dịch phải dựa trên chu kỳ này. |
| Thứ Tư > Thứ Bảy | "Office-worker hypothesis": khách mua sắm giờ nghỉ trưa, qua mobile, không phải shopping weekend. → Tối ưu mobile experience, gửi email/push notification giờ trưa. |

#### C. Promo Tổng hợp (Synthetic Dataset Pattern)

Calendar promo lặp theo nhịp **6-4-6-4 đều đặn**, chỉ có **2 mức discount**, và **80% không nhắm category cụ thể**.

| Insight | So What? |
|---|---|
| 6-4-6-4 pattern bất thường | Đây có thể là dataset tổng hợp (synthetic), không phải dữ liệu promo thực tế. Cần flag điều này trong báo cáo — ảnh hưởng đến độ tin cậy của phân tích promo. |
| Chỉ 2 mức discount có ý nghĩa | Đơn giản hóa chiến lược promo: chỉ dùng Percentage 10-20%, bỏ Fixed 50 VND (vô nghĩa về kinh tế). |
| 80% không category-targeted | Cơ hội lớn: áp dụng promo CÓ MỤC TIÊU cho Casual và GenZ trong mùa thấp điểm (Nov-Jan) để kích cầu. |

> **🎯 "So What?" Tổng:** Sửa hệ thống tái đặt hàng. Áp dụng lịch mùa vụ riêng (Apr-Jun peak). Đơn giản hóa promo, thêm category targeting. Tối ưu mobile experience (Wed lunch-time peak).

### 6.5 Hồi 4 & Cao trào — Dự báo & "So What?": "Nếu không làm gì, doanh thu tiếp tục giảm. Nhưng chúng ta có đòn bẩy."

#### Dự báo Baseline (Không can thiệp)

Mô hình LightGBM ensemble (R²=0.75, MAE=950K) dự báo: nếu không có thay đổi gì, doanh thu sẽ tiếp tục xu hướng giảm nhẹ, dao động quanh **~3.2-3.5 triệu VND/ngày** trong 2023-2024. Không sụp đổ thêm, nhưng cũng không phục hồi.

#### 5 Đòn bẩy Phục hồi (Recovery Simulator)

| Đòn bẩy | Impact (triệu/năm) | Effort | ROI | Ưu tiên |
|---|---|---|---|---|
| **1. Sửa Conversion Rate** (0.42% → 0.65%) | +300M | Thấp-Trung bình | ⭐⭐⭐⭐⭐ | **LÀM NGAY** |
| **2. Giảm Cancel Rate** (9.2% → 6%) | +200M | Thấp | ⭐⭐⭐⭐⭐ | **LÀM NGAY** |
| **3. Giảm Stockout Days** (1.16 → 0.5) | +150M | Trung bình-Cao | ⭐⭐⭐ | Q2-Q3 |
| **4. Tăng AOV** (+10%) | +100M | Trung bình | ⭐⭐⭐ | Q3 |
| **5. Tăng Sessions** (+20%) | +50M | Cao | ⭐ | Không ưu tiên |

> **🔑 Tại sao Conversion và Cancel là ưu tiên hàng đầu?**
>
> Vì đây là **đòn bẩy "sửa chữa" chứ không phải "xây mới"** — doanh nghiệp đã có sẵn traffic (11M sessions/năm) và đơn hàng (37K/năm). Chỉ cần **chuyển đổi tốt hơn những gì đang có**, không cần tạo ra nhu cầu mới.
>
> - **Conversion 0.42% → 0.65%**: Vẫn THẤP hơn đỉnh 2016 (1.0%). Đây là mục tiêu khiêm tốn, khả thi.
> - **Cancel 9.2% → 6%**: Vẫn CAO hơn chuẩn e-commerce VN (~4%). Còn dư địa cải thiện.
> - **Cả hai đều là low-effort**: Không cần thay đổi sản phẩm, không cần mở rộng thị trường, không cần đầu tư lớn.

> **🎯 "So What?" Tổng:** Ưu tiên 2 đòn bẩy đầu tiên — Conversion và Cancel reduction. Tổng tiềm năng: **+500 triệu VND/năm** chỉ từ việc sửa những thứ đang hỏng, không cần làm gì mới.

### 6.6 Kết — Lộ trình Hành động: "Bắt đầu từ đâu?"

| Quý | Hành động | Đòn bẩy | Kỳ vọng Impact |
|---|---|---|---|
| **Q1** | Audit UX checkout flow. Thêm size guide (giảm wrong_size returns). Triển khai win-back campaign cho At-Risk customers. | Conversion + Retention | +50M (ramp-up) |
| **Q2** | Triển khai COD→Prepay incentive (giảm 20% COD volume). Optimize mobile checkout (Wed lunch-time peak). | Cancel + Conversion | +120M |
| **Q3** | Triển khai 7-day reorder cycle. Tái cân bằng inventory: tăng stock cho GenZ, giảm cho Casual. | Stockout | +200M (cộng dồn) |
| **Q4** | Đánh giá và điều chỉnh. Mở rộng những gì hoạt động. Thêm category-targeted promos cho Casual+GenZ mùa thấp điểm. | Tối ưu liên tục | +350M (cộng dồn) |

> **Ngân sách đề xuất:** ~500 triệu VND cho cả năm (chủ yếu là nhân sự UX + tech). **ROI dự kiến:** 700-1,000% (thu về 3.5-5 tỷ từ khoản đầu tư 500 triệu).

---

### 6.7 Tóm tắt: Một Câu chuyện trong 60 Giây (Elevator Pitch)

> **"Doanh thu giảm 46% trong khi traffic tăng 67%. Không phải vì hết khách — khách vẫn đến, vẫn muốn mua. Họ chỉ không thể hoàn tất đơn hàng.**
>
> **Conversion rate giảm từ 1% xuống 0.4% — phễu bị thủng ở bước cuối cùng. AOV không đổi, basket size không đổi, margin không đổi. Vấn đề là VOLUME — và volume là thứ DỄ SỬA NHẤT nếu biết đúng chỗ hỏng.**
>
> **Cùng lúc đó, hệ thống tồn kho đang tự làm mình đau: 50% sản phẩm vừa hết hàng vừa tồn dư. Mùa cao điểm thực sự là tháng 4-6, không phải Tết. Thứ Tư bán chạy hơn Thứ Bảy — khách hàng mua sắm giờ nghỉ trưa trên điện thoại.**
>
> **Hai hành động đầu tiên: (1) Sửa conversion — audit UX checkout, tối ưu mobile; (2) Giảm cancel — chuyển COD sang trả trước. Tổng tiềm năng: +500 triệu/năm từ việc sửa những thứ đang hỏng. Không cần làm gì mới. Không cần thêm khách. Chỉ cần để khách mua hàng dễ dàng hơn."**

---

### 6.8 Bảng "So What?" — Từ Insight đến Hành động

| # | Insight Gốc | "So What?" — Hành động Cụ thể | Chủ sở hữu | Đo lường |
|---|---|---|---|---|
| 1 | Revenue -46%, Traffic +67% | **Dừng tăng chi marketing.** Phân bổ lại ngân sách sang cải thiện UX/Conversion. | CMO | CAC, ROAS |
| 2 | Conversion 1.0%→0.4% | **Audit toàn bộ checkout flow.** Tìm điểm friction (form fields, payment options, load time). | CTO/Product | Conversion rate |
| 3 | AOV ổn định, volume giảm | **Không thay đổi giá.** Tập trung vào việc tăng số đơn, không tăng giá trị đơn. | Pricing team | n_orders |
| 4 | 50.6% inventory paradox | **Chuyển chu kỳ tái đặt hàng từ 30→7-14 ngày.** Thay KPI fill_rate bằng stockout_days và lost_revenue. | Supply Chain | Stockout days |
| 5 | Apr-Jun peak, Wed>Sat | **Điều chỉnh lịch marketing & tồn kho.** Tập trung promo tháng 3-6. Gửi push notification giờ trưa thứ 3-4. | Marketing | Revenue/month |
| 6 | COD cancel rate 2-3× prepay | **Triển khai incentive chuyển COD→Prepay.** Ví dụ: freeship cho đơn trả trước, hoặc discount 2% cho prepay. | Ops/Finance | Cancel rate |
| 7 | Wrong size = #1 return reason | **Thêm size guide chi tiết cho Streetwear.** Bao gồm ảnh thực tế, số đo cụ thể, review có ảnh. | Product | Return rate |
| 8 | organic_search + referral = LTV cao nhất | **Tăng đầu tư SEO và referral program.** Giảm phụ thuộc vào email_campaign (high cancel). | Marketing | LTV by channel |
| 9 | Champions (25%) = 70% revenue | **Triển khai loyalty program.** Early access, VIP pricing, exclusive products cho Champions. | CRM | Retention rate |
| 10 | Model baseline: suy giảm chậm | **Forecast là "worst case scenario".** Mọi cải thiện vận hành sẽ đẩy thực tế vượt lên trên forecast. Dùng forecast làm baseline cho ROI calculation. | Strategy | Actual vs Forecast |

---

### 6.9 Nguyên tắc Data Storytelling cho Dự án Này

| Nguyên tắc | Cách áp dụng |
|---|---|
| **Bắt đầu bằng con số gây sốc** | "-46% revenue, +67% traffic" — một câu đủ để người nghe muốn biết thêm |
| **Mỗi chart kể MỘT ý** | Không nhồi nhét. Mỗi biểu đồ trả lời đúng 1 câu hỏi, dẫn đến 1 hành động |
| **Luôn trả lời "So What?"** | Không chỉ nói "conversion giảm" — phải nói "vì vậy cần audit UX, đây là việc CTO cần làm tuần sau" |
| **Kết nối các insight** | Inventory paradox ← ảnh hưởng → Stockout → ảnh hưởng → Conversion. Không insight nào đứng một mình |
| **Kết thúc bằng hành động cụ thể** | Không kết thúc bằng "cần cải thiện". Kết thúc bằng "Q1: làm X, chủ sở hữu Y, đo bằng Z" |
| **Ngôn ngữ kinh doanh, không phải kỹ thuật** | "Conversion rate" → "tỉ lệ khách hoàn tất mua hàng". "SHAP value" → "tầm quan trọng của từng yếu tố". |
| **Một câu chuyện, một slide deck** | Toàn bộ câu chuyện này có thể chuyển thành 10-12 slides cho board meeting

---
## Phần 7: Nguyên tắc Thiết kế & Khả năng Tái lập

### Nguyên tắc Thiết kế Trực quan hóa

| Nguyên tắc | Cách triển khai |
|---|---|
| **Bảng màu nhất quán** | Brand theme: nền `#0A1F1A`, xanh chính `#52B788`, `#FFA94D` cảnh báo, `#F25F5C` nguy hiểm, `#74C0FC` thông tin |
| **Tiêu đề biểu đồ ≤12 từ** | Mỗi tiêu đề trả lời "chuyện gì đã xảy ra" |
| **Chú thích trên điểm dữ liệu chính** | Mỗi biểu đồ có ≥1 chú thích chỉ vào anomaly hoặc phát hiện chính |
| **Dark theme nhất quán** | Tất cả biểu đồ dùng `plotly_dark` template hoặc matplotlib dark styling |
| **Responsive / container-width** | Biểu đồ Streamlit dùng `use_container_width=True` |
| **KPI cards glassmorphism** | `backdrop-filter: blur(12px)`, gradient backgrounds, hiệu ứng hover lift |
| **Phân cấp typography** | Outfit (display/headings) + Inter (body/metrics) + JetBrains Mono (số dạng bảng) |
| **Phân tích 4 cấp độ** | Mỗi câu chuyện có: Mô tả (cái gì) → Chẩn đoán (tại sao) → Dự báo (nếu...thì) → Đề xuất (làm gì) |

### Checklist Khả năng Tái lập

- [x] Random seed 42 (numpy + LightGBM + CatBoost + SHAP sampling)
- [x] TRAIN_CUTOFF = 2022-12-31 — tất cả phân tích an toàn leakage
- [x] Tất cả lag features dùng `shift(≥1)` — không có thông tin tương lai
- [x] `ord_count`, `ord_delivered` chỉ qua `lag≥365` (có sẵn cho test period)
- [x] TimeSeriesSplit CV với gap 14 ngày
- [x] SHAP tính trên mẫu training held-out
- [x] Tất cả hình lưu vào `reports/figures/`
- [x] Tất cả đường dẫn dữ liệu dùng `Path(__file__)` hoặc walk-up loop → ổn định với thay đổi working directory

### Thống kê Hình ảnh

| Thư mục | Số lượng | Notebook Nguồn |
|---|---|---|
| `reports/figures/fig_*` (profiling) | 6 | `00_data_profiling.ipynb` |
| `reports/figures/fig_*` (EDA) | 4 | `01_eda_exploratory.ipynb` |
| `reports/figures/A*` (forecast EDA) | 9 | `part3_pipeline.ipynb` |
| `reports/figures/D*` (CV) | 1 | `part3_pipeline.ipynb` |
| `reports/figures/E*` (SHAP) | 1 | `part3_pipeline.ipynb` |
| `reports/figures/G*` (forecast) | 1 | `part3_pipeline.ipynb` |
| Streamlit pages | 9 tương tác | `src/app/app_pages/*.py` |
| Hình bổ sung khác | 15+ | Các notebook khác |
| **Tổng số trực quan hóa** | **~50+** | |

---
## Phần 8: Hình ảnh Bổ sung Khác

Các hình dưới đây được tạo ra từ các notebook bổ sung (`frc_model_*.ipynb`, `train.ipynb`, `sales_forecasting_enhanced.ipynb`) và các phiên bản pipeline khác của Part 3.

### 7.1 `fig_abt_orders_dist.png` — Phân phối Orders trong ABT

<img src="../../reports/figures/fig_abt_orders_dist.png" alt="Phân phối orders đã enriched trong ABT" width="100%">

*Phân phối orders đã enriched trong ABT*


**Trực quan hóa cái gì:** Phân phối các biến trong `abt_orders_enriched.parquet` — net_revenue, discount_amount, unit_price.
**Plan:** Load `abt_orders_enriched.parquet` → multi-panel histograms với brand colors.
**Insight:** Xác nhận dữ liệu order đã được làm sạch và phân phối hợp lý trước khi đưa vào model.

### 7.2 `fig_shap_revenue.png` & `fig_forecast_revenue.png` — SHAP & Dự báo (phiên bản cũ)

<img src="../../reports/figures/fig_shap_revenue.png" alt="SHAP feature importance phiên bản đầu" width="100%">

*SHAP feature importance phiên bản đầu*


<img src="../../reports/figures/fig_forecast_revenue.png" alt="Dự báo doanh thu phiên bản đầu" width="100%">

*Dự báo doanh thu phiên bản đầu*


**Trực quan hóa cái gì:** Phiên bản sơ khai của SHAP analysis và forecast.
**Plan:** Phiên bản đầu của pipeline forecasting trước khi tinh chỉnh ensemble và feature engineering.
**Insight:** Các phiên bản đầu cho thấy mô hình đơn giản hơn với ít features hơn → cải thiện rõ rệt sau khi thêm 218 features.

### 7.3 `fig_lgbm_feature_importance.png` & `feature_importance_comparison.png` — So sánh Feature Importance

<img src="../../reports/figures/fig_lgbm_feature_importance.png" alt="LightGBM feature importance (gain-based)" width="100%">

*LightGBM feature importance (gain-based)*


<img src="../../reports/figures/feature_importance_comparison.png" alt="So sánh feature importance giữa các mô hình" width="100%">

*So sánh feature importance giữa các mô hình*


**Trực quan hóa cái gì:** Gain-based feature importance từ LightGBM và so sánh giữa các mô hình khác nhau (LGB-MAE vs LGB-Huber vs CatBoost).
**Plan:** `lgb.plot_importance()` cho gain importance, multi-model bar chart comparison.
**Insight:** LGB-MAE và LGB-Huber có ranking features khác nhau → ensemble blending giúp cân bằng. CatBoost feature importance tương tự LGB → nhất quán.

### 7.4 `eda_revenue_overview.png` & `eda_overview.png` — EDA Overview

<img src="../../reports/figures/eda_revenue_overview.png" alt="EDA — Tổng quan Doanh thu" width="100%">

*EDA — Tổng quan Doanh thu*


<img src="../../reports/figures/eda_overview.png" alt="EDA — Tổng quan toàn bộ" width="100%">

*EDA — Tổng quan toàn bộ*


**Trực quan hóa cái gì:** Tổng quan EDA — revenue trend, phân phối, anomaly detection.
**Plan:** Multi-panel overview charts từ giai đoạn EDA sớm, trước khi xây dựng ABT.
**Insight:** Xác nhận các phát hiện ban đầu: revenue collapse post-2016, seasonal pattern Apr-Jun, margin stability.

### 7.5 `eda_annual_growth.png` — Tăng trưởng Hàng năm

<img src="../../reports/figures/eda_annual_growth.png" alt="Tăng trưởng doanh thu hàng năm với YoY%" width="100%">

*Tăng trưởng doanh thu hàng năm với YoY%*


**Trực quan hóa cái gì:** Annual revenue bars với YoY growth rate annotations.
**Plan:** Groupby year → bar chart với text annotations hiển thị % thay đổi.
**Insight:** 2016→2017: -11%, 2017→2018: -10%, 2018→2019: -30% (cú sốc lớn nhất), 2019→2020: +5% (phục hồi nhẹ).

### 7.6 `eda_decomposition.png` & `acf_pacf.png` — Phân rã Chuỗi Thời gian

<img src="../../reports/figures/eda_decomposition.png" alt="Phân rã chuỗi thời gian — Trend, Seasonal, Residual" width="100%">

*Phân rã chuỗi thời gian — Trend, Seasonal, Residual*


<img src="../../reports/figures/acf_pacf.png" alt="ACF và PACF của chuỗi Revenue" width="100%">

*ACF và PACF của chuỗi Revenue*


**Trực quan hóa cái gì:** Seasonal decomposition (trend + seasonal + residual) và ACF/PACF correlograms.
**Plan:** `statsmodels.tsa.seasonal_decompose(period=365)` cho decomposition, `plot_acf()` + `plot_pacf()` cho correlogram.
**Insight:** ACF giảm chậm → non-stationary (cần differencing). PACF spike tại lag-1 → AR(1) component mạnh. Seasonal pattern rõ tại lag-7 và lag-365.

### 7.7 `eda_traffic_revenue.png` — Quan hệ Traffic-Doanh thu

<img src="../../reports/figures/eda_traffic_revenue.png" alt="Phân tích mối quan hệ giữa traffic và doanh thu" width="100%">

*Phân tích mối quan hệ giữa traffic và doanh thu*


**Trực quan hóa cái gì:** Sessions vs Revenue correlation analysis — scatter, time series overlay, lag correlation.
**Plan:** Merge web_traffic với sales → multi-panel correlation analysis.
**Insight:** Correlation Sessions-Revenue yếu dần theo thời gian (2013: r~0.5, 2022: r~0.15) — traffic không còn chuyển đổi thành doanh thu như trước. Xác nhận conversion collapse.

### 7.8 `cv_fold_results.png` & `fitted_2022.png` — CV & Fit Trực quan

<img src="../../reports/figures/cv_fold_results.png" alt="Kết quả Cross-Validation chi tiết" width="100%">

*Kết quả Cross-Validation chi tiết*


<img src="../../reports/figures/fitted_2022.png" alt="Giá trị fitted cho năm 2022" width="100%">

*Giá trị fitted cho năm 2022*


**Trực quan hóa cái gì:** Chi tiết CV folds và fitted values cho năm 2022.
**Plan:** Plot từng fold riêng biệt + fitted values overlay trên actual cho 2022.
**Insight:** Mô hình bám sát actual trong 2022 (R²=0.75) — tốt nhất vào các tháng có pattern rõ ràng, kém nhất ở các tháng chuyển mùa (Mar-Apr, Sep-Oct).

### 7.9 `shap_summary.png` & `shap_optimized.png` — SHAP Đầy đủ & Tối ưu

<img src="../../reports/figures/shap_summary.png" alt="SHAP summary plot đầy đủ" width="100%">

*SHAP summary plot đầy đủ*


<img src="../../reports/figures/shap_optimized.png" alt="SHAP analysis đã tối ưu" width="100%">

*SHAP analysis đã tối ưu*


**Trực quan hóa cái gì:** Phiên bản đầy đủ và đã tối ưu của SHAP analysis.
**Plan:** SHAP trên toàn bộ feature set → tối ưu hóa bằng cách loại bỏ features có importance thấp.
**Insight:** Sau khi tối ưu, top-50 features đã nắm bắt ~95% predictive power. Có thể giảm từ 218 xuống ~50 features mà không mất nhiều độ chính xác.

### 7.10 `forecast_final.png` & `forecast_optimized.png` — Dự báo Cuối cùng & Tối ưu

<img src="../../reports/figures/forecast_final.png" alt="Dự báo cuối cùng — Revenue + COGS cho 2023-2024" width="100%">

*Dự báo cuối cùng — Revenue + COGS cho 2023-2024*


<img src="../../reports/figures/forecast_optimized.png" alt="Dự báo đã tối ưu" width="100%">

*Dự báo đã tối ưu*


**Trực quan hóa cái gì:** Phiên bản đầy đủ và đã tối ưu của dự báo 2023-2024.
**Plan:** Dự báo với tất cả features vs chỉ top features → so sánh kết quả.
**Insight:** Dự báo với 50 features tối ưu cho kết quả gần như tương đương full model (MAE chênh <5%) → mô hình gọn hơn, nhanh hơn, dễ giải thích hơn.

---
**The GridBreakers · VinTelligence · VinUni DS&AI Club · Datathon 2026**

*Tài liệu này tổng hợp ~50+ trực quan hóa trên toàn bộ dự án Gridbreaker, bao gồm Data Profiling, EDA, Streamlit Dashboard, Dự báo & SHAP. Mỗi biểu đồ được phân tích theo 3 khía cạnh: mục đích (trực quan hóa cái gì), phương pháp (plan), và ý nghĩa kinh doanh (insight).*